# 1.6 Generalization bounds

A high-probability ceiling on the gap between the training error you measured and the true error you care about.

A finite-class bound reads as empirical risk plus a complexity penalty. It is useful because it separates what the model did on the sample from the uncertainty owed to class size, confidence, and sample size.

Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


def finite_penalty(H_size, delta, m):
    numerator = math.log(H_size) + math.log(1 / delta)
    return math.sqrt(numerator / (2 * m))


def finite_bound(emp, H_size, delta, m):
    penalty = finite_penalty(H_size, delta, m)
    return min(1.0, emp + penalty), penalty


def bound_rung(name, emp_values, H_size, delta, m_values):
    rows = []
    for m in m_values:
        for emp in emp_values:
            bound, penalty = finite_bound(emp, H_size, delta, m)
            rows.append({
                "name": name,
                "emp": float(emp),
                "H_size": int(H_size),
                "delta": float(delta),
                "m": int(m),
                "penalty": float(penalty),
                "bound": float(bound),
            })
    return rows

## The concept, built once (D1)

For a finite class, with probability at least $1-\delta$,

$$R(h)\le R_S(h)+\sqrt{rac{\ln|H|+\ln(1/\delta)}{2m}}.$$

The lesson computes the penalty for $|H|=100$, $\delta=0.05$, and $m=100$.

In [ ]:
lesson_bound, lesson_penalty = finite_bound(0.10, 100, 0.05, 100)
print("penalty:", lesson_penalty)
print("bound at empirical risk 0.10:", lesson_bound)

The plan's exact rounded penalty is $0.1949$. The same setup gives $0.0616$ at $m=1000$, $0.0195$ at $m=10000$, and $0.0975$ at $m=400$.

In [ ]:
assert np.isclose(round(lesson_penalty, 4), 0.1949)
assert np.isclose(round(finite_penalty(100, 0.05, 1000), 4), 0.0616)
assert np.isclose(round(finite_penalty(100, 0.05, 10000), 4), 0.0195)
assert np.isclose(round(finite_penalty(100, 0.05, 400), 4), 0.0975)
confidence_penalties = [finite_penalty(100, delta, 500) for delta in [0.10, 0.05, 0.01]]
assert np.allclose(np.round(confidence_penalties, 4), [0.0831, 0.0872, 0.0960])
print("confidence sweep:", np.round(confidence_penalties, 4))

## The dataset ladder

D1 is the lesson at $m=100$, D2 and D3 increase sample size to $1000$ and $10000$, D4 sweeps confidence at $m=500$, and D5 stresses a huge class with tiny sample size until the bound becomes vacuous.

In [ ]:
rungs = []
rungs.append(bound_rung("D1 m=100", [0.05, 0.10, 0.20], 100, 0.05, [100]))
rungs.append(bound_rung("D2 m=1000", [0.05, 0.10, 0.20], 100, 0.05, [1000]))
rungs.append(bound_rung("D3 m=10000", [0.05, 0.10, 0.20], 100, 0.05, [10000]))
rungs.append([
    {"name": "D4 confidence sweep", "emp": 0.10, "H_size": 100, "delta": delta, "m": 500, "penalty": finite_penalty(100, delta, 500), "bound": finite_bound(0.10, 100, delta, 500)[0]}
    for delta in [0.10, 0.05, 0.01]
])
rungs.append(bound_rung("D5 vacuous huge class", [0.05, 0.10, 0.20], 10 ** 12, 0.01, [10]))

for rows in rungs:
    print(rows[0]["name"])
    print("  rows=", len(rows))
    print("  preview=", rows[:3])

In [ ]:
summary = []
for rows in rungs:
    representative = rows[0]
    summary.append((representative["name"], representative["m"], representative["penalty"], representative["bound"]))
print("rung | m | penalty | first bound")
for name, m, penalty, bound in summary:
    print(f"{name}: {m} | {penalty:.4f} | {bound:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for ax, rows in zip(axes[0], rungs):
    labels = [str(idx + 1) for idx in range(len(rows))]
    emp = [row["emp"] for row in rows]
    penalty = [row["penalty"] for row in rows]
    ax.bar(labels, emp, label="empirical")
    ax.bar(labels, penalty, bottom=emp, label="penalty")
    ax.axhline(1.0, color="red", linestyle="--", linewidth=1)
    ax.set_ylim(0, max(1.1, max(np.asarray(emp) + np.asarray(penalty)) * 1.1))
    ax.set_title(rows[0]["name"])
    ax.grid(True, axis="y", alpha=0.3)
axes[0, 0].legend()
ms = np.array([100, 400, 1000, 10000])
penalties = np.array([finite_penalty(100, 0.05, int(m)) for m in ms])
axes[1, 0].plot(ms, penalties, marker="o")
axes[1, 0].set_xscale("log")
axes[1, 0].set_xlabel("sample size m")
axes[1, 0].set_ylabel("penalty")
axes[1, 0].set_title("Payoff curve: penalty vs sample size")
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on the hardest rung

Pitfall: vacuous bounds. If the penalty exceeds $1$, the clipped statement $R\le1$ is true but useless; the fix is to increase $m$ or use a tighter VC or Rademacher term.

In [ ]:
emp = 0.10
wrong_bound, wrong_penalty = finite_bound(emp, 10 ** 12, 0.01, 10)
fixed_bound, fixed_penalty = finite_bound(emp, 10 ** 12, 0.01, 10000)
print(f"wrong tiny-sample penalty={wrong_penalty:.3f}, bound={wrong_bound:.3f}")
print(f"fixed larger-sample penalty={fixed_penalty:.3f}, bound={fixed_bound:.3f}")
print(f"penalty improvement={wrong_penalty - fixed_penalty:.3f}")

## Evaluate it + Practice

- Metric: bound value and penalty.
- No-skill baseline: report empirical risk without a penalty.
- Cheap sanity check: quadrupling m should roughly halve the penalty.
- Ablation: set |H| to one and remove the capacity charge.
- Failure signals: penalty above one, clipped bounds, or finite-H formulas on infinite classes.

### Practice prompts

- Compute the penalty for |H|=1000, delta=0.05, m=500.

- Find m that makes the D1 penalty about half as large.

- Compare delta=0.1 and delta=0.001 at m=500.